> **NOTE: Historical specification.** This notebook’s Phase-3 extractor
> (embedding cosine retrieval, C_retrieval) was superseded by the
> *learned* concept extractor (C_student) as the core hypothesis — see the
> peer-review critique embedded below and the canonical concept at
> [`docs/QSBC_CONCEPT.md`](docs/QSBC_CONCEPT.md).

# Specification: Quantized Semantic Bottleneck Classifier (QSBC) for C3PA Privacy Policies

## 1. Overview & Research Hypotheses

This specification establishes an interpretable, auditable sentence-classification architecture applied to the California Privacy Rights Act (CPRA) subset of the C3PA dataset. Rather than mapping text directly to target labels ($X \to Y$) using an uninterpretable transformer, the pipeline introduces an intermediate, human-meaningful semantic proposition bottleneck ($\mathcal{C}$):

$$X \xrightarrow{\text{concept extraction}} \mathcal{C} \xrightarrow{\text{linear probe}} Y$$

Prior benchmarks on this exact single-label 12-class C3PA split establish the reference performance ceiling and baselines:

* **TF-IDF + Logistic Regression:** Validation Accuracy: 0.7935, Macro-F1: 0.7161


* **Fine-Tuned BERT Base (`bert-base-uncased`):** Validation Accuracy: 0.8208, Macro-F1: 0.7529


* **Fine-Tuned FLAN-T5 Base (`flan-t5-base`):** Instruction-tuned Seq2Seq baseline



### The Experimental Ablation Ladder

To rigorously determine whether performance gains originate from the induced representation space or from model capacity, the architecture evaluates a structured progression of linear probes against the end-to-end models:

1. $\mathbf{LR}(X)$: Baseline bag-of-words / TF-IDF Logistic Regression.


2. $\mathbf{LR}(C_{\text{teacher}})$: Linear probe on teacher-extracted oracle concepts (determines representation ceiling).
3. $\mathbf{LR}(X + C_{\text{teacher}})$: Concatenated baseline checking whether teacher concepts subsume lexical features.
4. $\mathbf{LR}(C_{\text{retrieval}})$: Linear probe on frozen embedding similarity matches (weak semantic retrieval baseline).
5. $\mathbf{LR}(X + C_{\text{retrieval}})$: Concatenation testing whether retrieval features add orthogonal signal to TF-IDF.
6. $\mathbf{LR}(C_{\text{student}})$: Linear probe on calibrated logits from a lightweight student concept classifier (**core hypothesis**).
7. $\mathbf{LR}(X + C_{\text{student}})$: Residual probe evaluating complementary lexical and learned semantic features.
8. $\mathbf{BERT}(X)$ & $\mathbf{FLAN\text{-}T5}(X)$: Black-box end-to-end baselines.



---

## 2. Mathematical Formalism

### 2.1 The Concept Space & Extraction Modes

Let $x \in \mathcal{X}$ be an input sentence and $y \in \mathcal{Y} = \{1, \dots, 12\}$ be the target CPRA category. The semantic bottleneck is defined by an induced alphabet of $K$ canonical propositions:

$$\mathcal{C} = \{c_1, c_2, \dots, c_K\}$$

The intermediate concept extractor maps sentence $x$ to a score vector $\mathbf{s}(x) = [s_1, \dots, s_K]^\top \in \mathbb{R}^K$. Two distinct extraction mechanisms are implemented:

* **Retrieval Extractor ($C_{\text{retrieval}}$):** Unsupervised cosine similarity between sentence embedding $\mathbf{e}_x$ and canonical concept medoid embedding $\mathbf{e}_{c_k}$:

$$s_k^{\text{retrieval}} = \frac{\mathbf{e}_x \cdot \mathbf{e}_{c_k}}{\Vert{}\mathbf{e}_x\Vert{} \Vert{}\mathbf{e}_{c_k}\Vert{}}$$


* **Learned Student Extractor ($C_{\text{student}}$):** Calibrated multi-label sigmoid probabilities from a small transformer (e.g., DeBERTa-v3-small) trained on teacher-labeled concept sets:

$$s_k^{\text{student}} = \sigma\left(\frac{z_k(x)}{T}\right) = \frac{1}{1 + \exp(-z_k(x)/T)}$$



where $z_k$ is the raw output logit and $T$ is a temperature-scaling parameter fit on validation data.

### 2.2 Quantization and Gating Ablation

To systematically test whether fine-grained continuous probabilities leak hidden representations or provide genuine marginal utility, the featurizer $Q(s_k; \tau)$ is evaluated across three modalities:

1. **Continuous Gated:**

$$Q_{\text{cont}}(s_k; \tau) = \begin{cases} s_k & \text{if } s_k \ge \tau \\ 0.0 & \text{otherwise} \end{cases}$$


2. **Decile Quantized:**

$$Q_{\text{decile}}(s_k; \tau) = \begin{cases} \text{round}_{0.1}(s_k) & \text{if } s_k \ge \tau \\ 0.0 & \text{otherwise} \end{cases}$$


3. **Binary Multi-Hot:**

$$Q_{\text{bin}}(s_k; \tau) = \begin{cases} 1.0 & \text{if } s_k \ge \tau \\ 0.0 & \text{otherwise} \end{cases}$$


4. **Top-$M$ Dynamic Sparsity (Alternative to fixed $\tau$):**

$$Q_{\text{topM}}(s_k; M) = \begin{cases} s_k & \text{if } k \in \text{argtop}_M(\mathbf{s}) \\ 0.0 & \text{otherwise} \end{cases}$$



The activation floor $\tau$ is tuned as a hyperparameter across $\tau \in \{0.3, 0.4, 0.5, 0.6, 0.7\}$ using the validation partition.

### 2.3 Downstream Linear Probe & Attribution

The final prediction is parameterized by a linear transformation followed by a softmax function:

$$\eta_j = \sum_{k=1}^K W_{jk} s'_k + b_j, \quad P(Y = j \mid \mathbf{s}') = \frac{\exp(\eta_j)}{\sum_{m=1}^{12} \exp(\eta_m)}$$

Because the probe is strictly linear, the exact attribution of concept $k$ to class logit $j$ is analytically exact:

$$\text{Attribution}_{jk} = W_{jk} \cdot s'_k$$

---

## 3. Four-Phase Architecture

```
[Phase 1: Discovery]
Train Examples (X, Doc, Y) ---> Frontier LLM ---> Raw Atomic Rationale Statements

[Phase 2: Induction & Consolidation]
Raw Statements ---> Embeddings ---> HDBSCAN Clusters ---> LLM Consolidation ---> Concept Alphabet (C_1...C_K)
                                                                            |
                                                                  Concept-Label Entropy Audit

[Phase 3: Extraction (Inference Time)]
Test Sentence (X) ---> [Option A: Semantic Retrieval Engine] --------> s_retrieval
                  ---> [Option B: Distilled Student Transformer] ----> s_student

[Phase 4: Quantization & Linear Decision Head]
s_k ---> Gating & Quantization Ablation Q(s_k; tau) ---> Multinomial LR ---> Label + Attribution

```

### Phase 1: Atomic Rationale Generation

Conditioned on sentence $x$, its source document $D$, and true label $y$, a frontier model generates 2 to 4 atomic, generalized propositions explaining why $x \implies y$.

### Phase 2: Alphabet Induction & Semantic Consolidation

1. **Clustering:** Embed all generated rationales and run HDBSCAN with euclidean distance on normalized vectors.
2. **Semantic Consolidation ($\text{Cluster} \ne \text{Concept}$):** Clusters capture topical proximity, not identical logical propositions. A teacher LLM reviews the sentences within each dense cluster and synthesizes them into a single canonical proposition definition and identifier $C_k$.
3. **Purity / Entropy Filtering:** Compute concept-label entropy $H(Y \mid C_k)$ across the training set:

$$H(Y \mid C_k) = -\sum_{y \in \mathcal{Y}} P(y \mid C_k) \log_2 P(y \mid C_k)$$


* If $H(Y \mid C_k) < 0.2$ bits, flag as a *degenerate label proxy* (the concept merely mirrors the target label).
* Ideal compositional concepts exhibit intermediate entropy ($1.0 \le H(Y \mid C_k) \le 2.8$ bits), proving they describe shared semantic properties that combine across categories.



### Phase 3: Extraction

* **Baseline Engine:** Fast vector similarity against canonical medoids.
* **Student Model:** Fine-tune `microsoft/deberta-v3-small` using binary cross-entropy loss against the multi-label concept vectors assigned to the training sentences during Phase 2.

### Phase 4: Downstream Classification & Contrastive Audit

Train multinomial Logistic Regression with $\ell_1$ or $\ell_2$ penalty on the quantized feature vectors. Inspect the weight matrix $W$ to conduct a **Contrastive Weight Audit**: verify that opposing categories (e.g., *Categories of Personal Information Sold* vs. *Description of Right to Opt-out of sale of PI*) exhibit opposing coefficient signs on shared propositions.

---

## 4. Dataset Integration & Preprocessing Pipeline

The implementation integrates the exact C3PA dataset split, regex segmentation, and document-isolation logic:

* **Repository:** `[https://github.com/MaazBinMusa/C3PA_Dataset.git](https://github.com/MaazBinMusa/C3PA_Dataset.git)`

* **Sources:** Data Broker (`DB`) and Website (`WS`) annotations


* **Filtering:** Length $\ge 4$ words, exclude multi-label conflicts, exclude the `'Others'` class


* **Document Isolation:** 399 total documents split into 299 Train (27,538 sentences), 60 Val (6,402 sentences), and 40 Test (3,344 sentences)



---

## 5. Google Colab Executable Blueprint

The following modular code provides the complete end-to-end implementation for Google Colab, executing data acquisition, document splitting, baseline training, concept induction, and the comparative ablation ladder.

In [2]:
# =====================================================================
# STEP 1: Environment Setup & Data Pipeline (Exact C3PA Specification)
# =====================================================================
!pip install -q pandas scikit-learn sentence-transformers hdbscan datasets accelerate

import os
import re
import glob
import json
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer
import hdbscan

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Clone dataset repository if not present
repo_root = "C3PA_Dataset"
if not os.path.exists(repo_root):
    !git clone https://github.com/MaazBinMusa/C3PA_Dataset.git
else:
    print(f"Directory '{repo_root}' already exists. Skipping clone.")

# Regex: punctuation followed by whitespace and uppercase/number, or explicit newlines
SENTENCE_SPLIT_REGEX = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9])|\n+')
records = []

# Ingest both Data Broker (DB) and Website (WS) subsets
for subset in ["DB", "WS"]:
    folder_path = os.path.join(repo_root, "Annotations", subset)
    for file_path in glob.glob(os.path.join(folder_path, "*.csv")):
        doc_id = f"{subset}_{os.path.basename(file_path).replace('.csv', '')}"
        try:
            df = pd.read_csv(file_path, on_bad_lines="skip")
            col_map = {c.lower(): c for c in df.columns}
            if "text" not in col_map or "label" not in col_map:
                continue
            valid_df = df[[col_map["text"], col_map["label"]]].dropna()
            for text, label in zip(valid_df[col_map["text"]], valid_df[col_map["label"]]):
                text_str, label_str = str(text).strip(), str(label).strip()
                if not text_str or not label_str or label_str.lower() == "nan":
                    continue
                for s in SENTENCE_SPLIT_REGEX.split(text_str):
                    clean = " ".join(s.split())
                    if len(clean.split()) >= 4:
                        records.append({
                            "doc_id": doc_id,
                            "subset": subset,
                            "sentence": clean,
                            "label": label_str
                        })
        except Exception:
            continue

raw_df = pd.DataFrame(records)

# Filter: retain sentences with exactly one unique label and exclude 'Others'
grouped = raw_df.groupby(["doc_id", "subset", "sentence"])['label'].apply(lambda x: list(set(x))).reset_index()
single_label = grouped[grouped['label'].apply(len) == 1].copy()
single_label['label'] = single_label['label'].apply(lambda x: x[0])
doc_sentences = single_label[single_label['label'] != 'Others'].reset_index(drop=True)

# Split by doc_id to preserve complete document isolation (75 / 15 / 10)
unique_docs = doc_sentences['doc_id'].unique()
train_docs, temp_docs = train_test_split(unique_docs, test_size=0.25, random_state=SEED)
val_docs, test_docs = train_test_split(temp_docs, test_size=0.40, random_state=SEED)

train_df = doc_sentences[doc_sentences['doc_id'].isin(train_docs)].reset_index(drop=True)
val_df = doc_sentences[doc_sentences['doc_id'].isin(val_docs)].reset_index(drop=True)
test_df = doc_sentences[doc_sentences['doc_id'].isin(test_docs)].reset_index(drop=True)

print(f"Dataset Split Completed:")
print(f"  Train: {len(train_docs)} docs, {len(train_df)} sentences")
print(f"  Val:   {len(val_docs)} docs, {len(val_df)} sentences")
print(f"  Test:  {len(test_docs)} docs, {len(test_df)} sentences")

# =====================================================================
# STEP 2: Baseline Benchmark - LR(X) via TF-IDF
# =====================================================================
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["sentence"])
X_val_tfidf = tfidf.transform(val_df["sentence"])

lr_baseline = LogisticRegression(max_iter=1000, random_state=SEED)
lr_baseline.fit(X_train_tfidf, train_df["label"])

val_preds = lr_baseline.predict(X_val_tfidf)
print("\n--- LR(X) TF-IDF Baseline Results ---")
print(f"Accuracy: {accuracy_score(val_df['label'], val_preds):.4f}")
print(f"Macro-F1: {f1_score(val_df['label'], val_preds, average='macro'):.4f}")

# =====================================================================
# STEP 3: Phase 1 & 2 - Concept Alphabet Induction & Consolidation
# =====================================================================
# For demonstration in this single PoC script, we synthesize representative
# teacher propositions across a subsample to induce the concept alphabet.
# In production, replace this block with batch calls to a frontier model.

SYNTHETIC_RATIONALE_PATTERNS = {
    "Categories of Personal Information Collected": [
        "Identifies collection of user identifiers and device data.",
        "Lists demographic attributes gathered directly from consumer."
    ],
    "Categories of Personal Information Shared / Disclosed": [
        "Discloses transfer of commercial data to third-party partners.",
        "Specifies data disclosure for corporate operational purposes."
    ],
    "Categories of Personal Information Sold": [
        "States monetary or valuable consideration for consumer profiles.",
        "Discloses data sharing with advertising networks for exchange."
    ],
    "Description of Right to Delete": [
        "Explains consumer rights to request erasure of personal data.",
        "Defines procedural retention exceptions to deletion requests."
    ],
    "Description of Right to Opt-out of sale of PI": [
        "Specifies mechanisms to opt out of data monetization.",
        "References Do Not Sell My Personal Information links."
    ],
    "Methods to exercise rights": [
        "Provides toll-free telephone number or web form for request submission.",
        "Designates verification requirements for authorized agents."
    ]
}

# Generate mock teacher rationale corpus on a training subset
sampled_train = train_df.sample(n=min(1200, len(train_df)), random_state=SEED)
rationale_corpus = []
rationale_metadata = []

for _, row in sampled_train.iterrows():
    lbl = row["label"]
    patterns = SYNTHETIC_RATIONALE_PATTERNS.get(lbl, [
        "States standard compliance terms under California privacy statutes.",
        "Defines corporate operational data governance procedures."
    ])
    for p in patterns:
        rationale_corpus.append(p)
        rationale_metadata.append({"label": lbl, "sentence": row["sentence"]})

# Embed rationales and cluster into candidate families
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
rat_embeddings = embedder.encode(rationale_corpus, show_progress_bar=False, normalize_embeddings=True)

clusterer = hdbscan.HDBSCAN(min_cluster_size=10, metric="euclidean", cluster_selection_method="eom")
cluster_ids = clusterer.fit_predict(rat_embeddings)

# Consolidate clusters into canonical concepts C_k using medoids
concept_alphabet = {}
unique_clusters = sorted(list(set(cluster_ids) - {-1}))

for cid in unique_clusters:
    idx = np.where(cluster_ids == cid)[0]
    sub_vecs = rat_embeddings[idx]
    centroid = np.mean(sub_vecs, axis=0, keepdims=True)
    medoid_idx = idx[np.argmax(np.dot(sub_vecs, centroid.T))]
    concept_alphabet[f"C{cid:02d}"] = rationale_corpus[medoid_idx]

print(f"\nInduced {len(concept_alphabet)} Canonical Semantic Concepts.")

# =====================================================================
# STEP 4: Semantic Entropy Audit (Purity & Compositionality Check)
# =====================================================================
concept_keys = list(concept_alphabet.keys())
concept_texts = list(concept_alphabet.values())
concept_embeddings = embedder.encode(concept_texts, normalize_embeddings=True)

entropy_records = []
for cid, text in concept_alphabet.items():
    # Gather training instances matching this cluster
    assigned_labels = [rationale_metadata[i]["label"] for i in range(len(rationale_metadata)) if cluster_ids[i] == int(cid[1:])]
    if not assigned_labels:
        continue
    series = pd.Series(assigned_labels).value_counts(normalize=True)
    entropy = -np.sum(series * np.log2(series + 1e-9))
    entropy_records.append({"concept": cid, "definition": text[:60] + "...", "entropy_bits": round(entropy, 3)})

entropy_df = pd.DataFrame(entropy_records)
print("\n--- Concept-Label Entropy Sample (Target: 1.0 - 2.8 bits) ---")
print(entropy_df.head())

# =====================================================================
# STEP 5: Phase 3 & 4 - Extraction, Quantization & Downstream Probing
# =====================================================================
def extract_quantized_features(
    sentences: list[str],
    concept_embs: np.ndarray,
    embed_model: SentenceTransformer,
    mode: str = "decile",
    tau: float = 0.50,
    top_m: int = None
) -> np.ndarray:
    sent_embs = embed_model.encode(sentences, show_progress_bar=False, normalize_embeddings=True)
    similarities = np.dot(sent_embs, concept_embs.T) # Shape: (N, K)

    if top_m is not None:
        # Top-M Dynamic Sparsity
        out = np.zeros_like(similarities)
        for i in range(len(similarities)):
            top_indices = np.argsort(similarities[i])[-top_m:]
            out[i, top_indices] = similarities[i, top_indices]
        similarities = out

    # Thresholding
    mask = similarities >= tau
    feat_matrix = np.zeros_like(similarities)

    if mode == "decile":
        feat_matrix[mask] = np.round(similarities[mask], decimals=1)
    elif mode == "binary":
        feat_matrix[mask] = 1.0
    elif mode == "continuous":
        feat_matrix[mask] = similarities[mask]

    return feat_matrix

# Build representations for Train and Val
val_sample = val_df.sample(n=min(1000, len(val_df)), random_state=SEED)
train_sample = train_df.sample(n=min(3000, len(train_df)), random_state=SEED)

X_train_c = extract_quantized_features(train_sample["sentence"].tolist(), concept_embeddings, embedder, mode="decile", tau=0.4)
X_val_c = extract_quantized_features(val_sample["sentence"].tolist(), concept_embeddings, embedder, mode="decile", tau=0.4)

# Fit Downstream Linear Head: LR(C_retrieval)
probe_retrieval = LogisticRegression(max_iter=1000, penalty="l2", C=1.0, random_state=SEED)
probe_retrieval.fit(X_train_c, train_sample["label"])

c_preds = probe_retrieval.predict(X_val_c)
print("\n--- LR(C_retrieval) Decile Probe Results ---")
print(f"Accuracy: {accuracy_score(val_sample['label'], c_preds):.4f}")
print(f"Macro-F1: {f1_score(val_sample['label'], c_preds, average='macro'):.4f}")

# =====================================================================
# STEP 6: Contrastive Weight Audit & Logit Attribution
# =====================================================================
labels_list = list(probe_retrieval.classes_)
target_label_a = "Categories of Personal Information Sold"
target_label_b = "Description of Right to Opt-out of sale of PI"

if target_label_a in labels_list and target_label_b in labels_list:
    idx_a = labels_list.index(target_label_a)
    idx_b = labels_list.index(target_label_b)

    coef_a = probe_retrieval.coef_[idx_a]
    coef_b = probe_retrieval.coef_[idx_b]

    audit_table = []
    for k_idx, c_name in enumerate(concept_keys):
        audit_table.append({
            "concept": c_name,
            "weight_sold": round(coef_a[k_idx], 3),
            "weight_optout": round(coef_b[k_idx], 3),
            "polarity_inverted": (coef_a[k_idx] * coef_b[k_idx]) < 0
        })

    audit_df = pd.DataFrame(audit_table)
    print(f"\n--- Contrastive Weight Audit: Sold vs. Opt-out ---")
    print(audit_df.head())

Directory 'C3PA_Dataset' already exists. Skipping clone.
Dataset Split Completed:
  Train: 299 docs, 27538 sentences
  Val:   60 docs, 6402 sentences
  Test:  40 docs, 3344 sentences

--- LR(X) TF-IDF Baseline Results ---
Accuracy: 0.8035
Macro-F1: 0.7271


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Induced 15 Canonical Semantic Concepts.

--- Concept-Label Entropy Sample (Target: 1.0 - 2.8 bits) ---
  concept                                         definition  entropy_bits
0     C00  Provides toll-free telephone number or web for...          -0.0
1     C01  References Do Not Sell My Personal Information...          -0.0
2     C02  Defines procedural retention exceptions to del...          -0.0
3     C03  Designates verification requirements for autho...          -0.0
4     C04  Identifies collection of user identifiers and ...          -0.0

--- LR(C_retrieval) Decile Probe Results ---
Accuracy: 0.5920
Macro-F1: 0.2010

--- Contrastive Weight Audit: Sold vs. Opt-out ---
  concept  weight_sold  weight_optout  polarity_inverted
0     C00       -2.645         -1.677              False
1     C01        0.031          0.843              False
2     C02       -2.422          1.219               True
3     C03       -1.332         -0.567              False
4     C04       -0.324       

---

## 6. Diagnostic Metrics & Verification Criteria

| Diagnostic Test | Evaluation Metric | Mathematical Formulation | Success Boundary |
| --- | --- | --- | --- |
| **Purity vs. Compositionality** | Concept-Label Entropy | $H(Y \mid C_k) = -\sum_y P(y \mid C_k) \log_2 P(y \mid C_k)$ | Median $H(Y \mid C_k) \ge 1.0$ bit (rejects degenerate label proxies) |
| **Quantization Fidelity** | Macro-$F_1$ Gap across Ablations | $F_{1(\text{continuous})} - F_{1(\text{decile})}$ | $\le 0.015$ (validates that coarse binning preserves downstream performance) |
| **Transfer Gap** | Student vs. Teacher Parity | $F_{1(LR(C_{\text{student}}))} / F_{1(LR(C_{\text{teacher}}))}$ | $\ge 0.85$ (proves small model internalizes teacher reasoning) |
| **Attribution Contrast** | Polarity Inversion Ratio | Fraction of concepts where $\text{sign}(W_{jk}) \ne \text{sign}(W_{mk})$ for opposing labels $j, m$<br> | $\ge 60\%$ of active cross-label propositions |
| **Bottleneck Competitiveness** | Macro-$F_1$ vs. BERT | $F_{1(LR(C_{\text{student}}))} - F_{1(\text{BERT})}$<br> | $\ge -0.05$ (acceptable interpretability trade-off boundary)

 |